_Date: 2026-09-08_


# Amplitude Rabi Chevron -- Custom 8 ns Gaussian Pulse (Zero-Padded)

2D sweep of drive **frequency** x drive **amplitude**, using the `contrib` experiment
`amplitude_rabi_chevron` from `laboneq_applications` -- same as `2026_08_11_Amplitude_Rabi.ipynb`,
except the drive pulse is replaced by a **custom 8 ns Gaussian pulse**: 8 ns of zero
padding followed by 8 ns of Gaussian envelope, designed sample-by-sample (see Section 1a).


In [ ]:
import numpy as np
from laboneq.simple import *

from laboneq_applications.qpu_types.tunable_transmon import (
    TunableTransmonQubit,
    TunableTransmonOperations,
)
from laboneq_applications.contrib.experiments import amplitude_rabi_chevron

## 1. Device Setup & QPU -- SHFQC (SG drive / QA readout)

Single qubit `q0`, driven on an SG channel, read out on a QA channel of the same SHFQC.
Built via the programmatic `DeviceSetup` API (`add_instruments`/`add_connections`),
matching ZI's official `getting_started_shfqc_plus` example -- `DeviceSetup.from_descriptor()`'s
YAML grammar only supports separate `SHFSG`/`SHFQA` entries, not the combined `SHFQC`
instrument type, and raises `AssertionError: Invalid device type` from the emulator if
you try `SHFQC:` as a descriptor key.

**Fill in before running on real hardware:**
- `address` -- your SHFQC's device serial (e.g. `dev12345`, check the LabOne Web UI or the
  label on the front panel).
- `device_options` -- your SHFQC's installed options string (LabOne Web UI -> Device tab,
  or `*OPT?`). Must contain a channel-count token (`QC2CH`/`QC4CH`/`QC6CH`) matching your
  instrument's SG channel count; drop `/PLUS/16W` if you don't have those options.
- `host` -- the LabOne dataserver host (`"localhost"` if running on the same PC as the
  dataserver).
- Every `qubit.parameters.*` value below -- these are placeholders. Use your qubit's
  already-calibrated values (e.g. from your 1D `amplitude_rabi` run) rather than these.

`do_emulation=True` is kept as the default below so nothing is sent to the instrument
until you've checked the values above -- flip to `False` once confirmed.


In [11]:
device_setup = DeviceSetup(uid="shfqc_setup")
device_setup.add_dataserver(host="localhost", port="8004")

shfqc = SHFQC(
    uid="device_shfqc",
    address="dev12073",
    interface="1GbE",
    device_options="SHFQC/LRT/PLUS/QC6CH/RTR",
    reference_clock_source="internal",
)
device_setup.add_instruments(shfqc)

device_setup.add_connections(
    "device_shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT", type="iq"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT", type="iq"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT", type="acquire"),
)

q_uid = "q0"
qubits = TunableTransmonQubit.from_device_setup(device_setup)
qubit = next(q for q in qubits if q.uid == q_uid)

# TODO: replace every value below with your qubit's real calibrated parameters
qubit.parameters.resonance_frequency_ge = 5.0e9        # qubit ge transition frequency
qubit.parameters.drive_lo_frequency = 4.8e9            # SG channel LO frequency
qubit.parameters.readout_resonator_frequency = 7.0e9   # readout resonator frequency
qubit.parameters.readout_lo_frequency = 6.8e9          # QA channel LO frequency
qubit.parameters.ge_drive_amplitude_pi = 0.8           # calibrated pi-pulse amplitude
qubit.parameters.ge_drive_amplitude_pi2 = 0.4          # calibrated pi/2-pulse amplitude

qpu = QPU(quantum_elements=qubits, quantum_operations=TunableTransmonOperations())

session = Session(device_setup)
session.connect(do_emulation=False, ignore_version_mismatch=True)   # set False once the address/options/parameters above are confirmed


[2026.09.08 15:00:06.204] INFO    Logging initialized from [Default inline config in laboneq.laboneq_logging] logdir is /Users/avishekc/Library/CloudStorage/OneDrive-ZurichInstrumentsAG/Documents/ZI Supports/MECH_Qubit/laboneq_output/log
[2026.09.08 15:00:06.214] INFO    VERSION: laboneq 26.7.0
[2026.09.08 15:00:06.216] INFO    Connecting to data server at localhost:8004
[2026.09.08 15:00:06.234] INFO    Connected to Zurich Instruments LabOne Data Server version 26.07.2.5 at localhost:8004
[2026.09.08 15:00:06.242] INFO    Configuring the device setup
[2026.09.08 15:00:06.247] INFO    The device setup is configured


## 1a. Custom 8 ns Gaussian Pulse (Zero-Padded)

A plain `playWave`-style pulse -- which is what `x180`/`rx` compiles down to -- has a
hard **32-sample (16 ns @ 2 GSa/s) minimum length** on the SHFQC's SG channels (only the
lower-level command-table mechanism reaches the 16-sample/8 ns floor, and that doesn't go
through the normal pulse DSL at all -- see `SG_CommandTable_8ns_Pulse.ipynb`).

So the pulse here is **16 ns total** (32 samples), split into two 8 ns halves designed
sample-by-sample: the first 16 samples are zero, the next 16 form a Gaussian envelope.
This lands exactly on the 32-sample/playWave minimum with no compiler padding.

No operation override is needed -- `rx()` builds its pulse from
`qubit.parameters.ge_drive_pulse` (a dict naming a registered `pulse_library` functional)
and `qubit.parameters.ge_drive_length`. Pointing those at the pulse below means the
**unmodified** library `x180`/`rx` automatically plays it, so this stays independent of
the marker on/off toggle in Section 2a.


In [12]:
SIGMA_SAMPLES = 16 / 6   # ~3-sigma across the 8 ns Gaussian half -- adjustable


@pulse_library.register_pulse_functional
def gaussian_zeropad_8ns(x, **_):
    n = len(x)            # total samples for whatever `length` was requested
    half = n // 2          # first half = zero pad, second half = Gaussian
    envelope = np.zeros(n)
    for i in range(half):                        # sample-by-sample, as requested
        n_rel = i - (half - 1) / 2                # centered index within the Gaussian half
        envelope[half + i] = np.exp(-0.5 * (n_rel / SIGMA_SAMPLES) ** 2)
    return envelope


qubit.parameters.ge_drive_pulse = {"function": "gaussian_zeropad_8ns"}
qubit.parameters.ge_drive_length = 16e-9   # 32 samples @ 2 GSa/s -- 16 zero + 16 Gaussian


## 2. Frequency x Amplitude Sweep

- **Frequency**: +/-20 MHz around the qubit's calibrated `resonance_frequency_ge`, 41 points.
- **Amplitude**: 0 to 1 (full drive-amplitude scale), 21 points.

Adjust `N_FREQ`/`N_AMP` or the frequency span to trade resolution against run time
(total real-time shots ~= `N_FREQ * N_AMP * count`).


In [13]:
N_FREQ = 41
N_AMP = 21
FREQ_SPAN = 20e6   # +/- around resonance_frequency_ge

f_center = qubit.parameters.resonance_frequency_ge
frequencies = f_center + np.linspace(-FREQ_SPAN, FREQ_SPAN, N_FREQ)
amplitudes = np.linspace(0, 1, N_AMP)

print(f"Sweeping {N_FREQ} frequencies around {f_center/1e9:.4f} GHz, {N_AMP} amplitudes 0-1")


Sweeping 41 frequencies around 5.0000 GHz, 21 amplitudes 0-1


## 2a. Add a Marker to the Drive Pulse

`amplitude_rabi_chevron`'s `create_experiment` plays the drive pulse via `qop.x180(q,
amplitude=amplitude)`, which delegates to `rx()` -- there's no `marker=` argument exposed
on `x180`/`rx`, so a marker can't be passed in through the workflow call itself.

Instead, `rx` is overridden on `qpu.quantum_operations` (the same override mechanism
`laboneq_applications` itself uses to register operations) so it plays with
`marker={"marker1": {"enable": True}}` every time -- `x180` (and therefore the chevron's
drive pulse) calls `self.rx(...)` internally, so this applies automatically without
touching the installed library. `marker1` is enabled for the duration of the drive pulse
on every sweep point, so a scope can trigger on it in sync with each drive pulse, same as
the chirp pulse notebook.

This override still uses `params["pulse"]`/`params["length"]` from the qubit, so it plays
the custom 8 ns pulse from Section 1a with the marker added -- the two features compose
independently.


In [14]:
@qpu.quantum_operations.register
def rx(self, q, angle, transition=None, amplitude=None, phase=0.0,
       increment_oscillator_phase=None, length=None, pulse=None) -> None:
    drive_line, params = q.transition_parameters(transition)
    if transition == "ef":
        dsl.active_section().on_system_grid = True
    if amplitude is None:
        amplitude = (angle / np.pi) * params["amplitude_pi"]
    if length is None:
        length = params["length"]
    rx_pulse = dsl.create_pulse(params["pulse"], pulse, name="rx_pulse")
    dsl.play(
        q.signals[drive_line],
        amplitude=amplitude,
        phase=phase,
        increment_oscillator_phase=increment_oscillator_phase,
        length=length,
        pulse=rx_pulse,
        marker={"marker1": {"enable": True}},
    )


## 2b. Arm the SHFQC Scope for a Triggered Capture (zhinst.toolkit)

Same architecture question as the earlier `SHFQC_QA_Scope_Capture.ipynb`: LabOne Q's
`Experiment` DSL has no Scope-module concept (`AcquisitionType.RAW` is a different thing,
tied to the acquire/integration pipeline) -- the Scope block itself is only reachable via
`session.devices[...]`, which returns the *same* `zhinst.toolkit` device object toolkit
code uses directly.

This time the capture is **triggered**, not free-run: `trigger_input` is set to
`channel0_sequencer_monitor0` -- the QA channel's own sequencer-generated trigger, which
fires every time the compiled experiment plays a readout pulse. That's a real trigger
source only while a compiled experiment is actively driving that sequencer, which Section
3 below does.

`shfqc_device.scopes[0].run(single=True)` only arms the scope and confirms the `enable`
flag flipped (confirmed from the real `zhinst-toolkit` source, `SHFScope.run()`) -- it does
**not** wait for an actual trigger/capture. So this cell must run *before* Section 3, and
the fetch (Section 3a below) happens *after* -- same "arm first, then trigger the source"
ordering used by every scope capture in this session's other notebooks. The Scope block is
a separate, passive hardware unit from the QA sequencer's uploaded program, so arming it
here doesn't conflict with Section 3's compiled experiment owning that sequencer.


In [ ]:
CHANNEL_INDEX = 0   # matches q0's QACHANNELS/0 readout, Section 1
SCOPE_CAPTURE_DURATION = 2e-6   # s
SHFQA_SCOPE_SAMPLING_RATE = 2e9   # fixed, Hz
SCOPE_NUM_SAMPLES = int(SCOPE_CAPTURE_DURATION * SHFQA_SCOPE_SAMPLING_RATE)

shfqc_device = session.devices["device_shfqc"]   # same toolkit object LabOne Q uses internally

shfqc_device.scopes[0].configure(
    input_select={0: f"channel{CHANNEL_INDEX}_signal_input"},
    num_samples=SCOPE_NUM_SAMPLES,
    trigger_input=f"channel{CHANNEL_INDEX}_sequencer_monitor0",   # fires when the compiled experiment plays a readout pulse
    num_segments=1,
    num_averages=1,
    trigger_delay=0.0,
)
shfqc_device.scopes[0].run(single=True)   # arms and confirms armed -- does NOT wait for a trigger
print("Scope armed -- run Section 3 next to trigger it, then fetch with Section 3a.")


## 3. Run the Chevron Experiment


In [21]:
options = amplitude_rabi_chevron.experiment_workflow.options()
options.count(1024)
options.use_cal_traces(True)

chevron_result = amplitude_rabi_chevron.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=q_uid,
    frequencies=frequencies,
    amplitudes=amplitudes,
    options=options,
).run()

chevron_result.tasks


[2026.09.08 15:11:52.962] INFO     ────────────────────────────────────────────────────────────────────────────── 
[2026.09.08 15:11:52.963] INFO      Workflow 'amplitude_rabi_chevron': execution started at 2026-09-08            
[2026.09.08 15:11:52.963] INFO      13:11:52.962229Z                                                              
[2026.09.08 15:11:52.964] INFO     ────────────────────────────────────────────────────────────────────────────── 
[2026.09.08 15:11:52.964] INFO    Task 'temporary_qpu': started at 2026-09-08 13:11:52.964590Z
[2026.09.08 15:11:52.965] INFO    Task 'temporary_qpu': ended at 2026-09-08 13:11:52.965408Z
[2026.09.08 15:11:52.966] INFO    Task 'temporary_quantum_elements_from_qpu': started at 2026-09-08 
[2026.09.08 15:11:52.966] INFO    13:11:52.966042Z
[2026.09.08 15:11:52.967] INFO    Task 'temporary_quantum_elements_from_qpu': ended at 2026-09-08 13:11:52.967161Z
[2026.09.08 15:11:52.967] INFO    Task 'create_experiment': started at 2026-09-08 13:

TaskResult(name=temporary_qpu, index=()), TaskResult(name=temporary_quantum_elements_from_qpu, index=()), TaskResult(name=create_experiment, index=()), TaskResult(name=compile_experiment, index=()), TaskResult(name=run_experiment, index=()), WorkflowResult(name=analysis_workflow, index=())

## 3a. Fetch the Triggered Scope Capture

Reads back whatever the scope captured on its first `channel0_sequencer_monitor0`
trigger -- the very first readout pulse played during Section 3's sweep (shot 0 of the 2D
frequency x amplitude sweep), not a specific chosen sweep point.


In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

scope_data, *_ = shfqc_device.scopes[0].read()
raw_scope = np.atleast_2d(scope_data[0])[0]   # normalize in case the segment axis was squeezed away
t_scope = np.arange(len(raw_scope)) / SHFQA_SCOPE_SAMPLING_RATE * 1e6   # us

fig_scope = go.Figure()
fig_scope.add_trace(go.Scatter(x=t_scope, y=raw_scope.real, name="I", line=dict(color="steelblue", width=0.8)))
fig_scope.add_trace(go.Scatter(x=t_scope, y=raw_scope.imag, name="Q", line=dict(color="darkorange", width=0.8)))
fig_scope.update_xaxes(title_text="Time (us)")
fig_scope.update_yaxes(title_text="Amplitude (a.u.)")
fig_scope.update_layout(title="Triggered raw QA scope capture (first readout pulse)", template="plotly_white",
                         paper_bgcolor="white", plot_bgcolor="white")
fig_scope.show()


## 3b. Pulse Sheet Viewer

Renders the compiled experiment's pulse sequence (drive/measure/reset, per sweep step)
as an interactive HTML file, then force-opens it in your **system default browser** --
the file is a self-contained ~1.4 MB JS bundle, not a CDN-loaded chart, so VS Code's
notebook link/preview can't execute it properly (shows raw text instead of the chart).

This is the natural place to visually confirm the zero-pad + Gaussian shape from
Section 1a actually compiled as intended -- zoom into one drive pulse and check for the
8 ns flat/zero lead-in before the Gaussian.

The full 2D sweep has `N_FREQ * N_AMP` real-time steps -- with the current 41x21 settings
that's 861, which can make the pulse sheet large/slow to render and may hit
`max_events_to_publish` truncation (increased below, but raise further if you still see a
truncation warning). For a quick structural sanity check instead of the full sweep, drop
`N_FREQ`/`N_AMP` to e.g. 2x2 before running Section 3.


In [8]:
import glob
import os
import webbrowser

compiled_exp = chevron_result.tasks["compile_experiment"].output
show_pulse_sheet("chevron_pulse_sheet", compiled_exp, max_events_to_publish=5000)

# Force-open in the system browser -- VS Code's notebook viewer can't run the pulse
# sheet's embedded JS bundle, so the IPython link/preview shows raw text instead.
latest_pulse_sheet = max(glob.glob("chevron_pulse_sheet_*.html"), key=os.path.getmtime)
webbrowser.open(f"file://{os.path.abspath(latest_pulse_sheet)}")
print(f"Opened {latest_pulse_sheet} in your default browser")


[2026.09.08 14:45:54.688] INFO    Recompiling the experiment due to missing extra information in the compiled experiment. Compile with `OUTPUT_EXTRAS=True` and `MAX_EVENTS_TO_PUBLISH=5000` to bypass this step with a small impact on the compilation time.
[2026.09.08 14:45:54.689] INFO    Starting LabOne Q Compiler run...
[2026.09.08 14:45:54.690] INFO    Resolved modulation type of oscillator on signal: 'q0/acquire' to Software
[2026.09.08 14:45:54.690] INFO    Resolved modulation type of oscillator on signal: 'q0/drive' to Hardware
[2026.09.08 14:45:54.690] INFO    Resolved modulation type of oscillator on signal: 'q0/measure' to Software
[2026.09.08 14:45:54.707] INFO    Schedule completed. [0.016 s]
[2026.09.08 14:45:54.778] INFO    Code generation completed. [0.058 s]
[2026.09.08 14:45:54.787] INFO    Completed compilation step 1 of 1. [0.096 s]
[2026.09.08 14:45:54.793] INFO    Finished LabOne Q Compiler run.
[2026.09.08 14:45:54.794] WARNING Pulse sheet viewer: The event list was 

## 4. Inspect Results

`analysis_workflow` (run automatically above via `options.do_analysis`, default `True`)
already plots the 2D chevron pattern. This cell pulls the raw acquired data directly, in
case you want a custom plot instead.


In [9]:
acquired_data = chevron_result.tasks["run_experiment"].output
raw = acquired_data[q_uid].result
print("Raw result array shape (amplitude x frequency):", np.shape(raw))


Raw result array shape (amplitude x frequency): ()
